<a href="https://www.kaggle.com/code/avikdas567/maze-crawler?scriptVersionId=318723385" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Maze Crawler — Adaptive Fog-of-War Agent

This notebook builds a self-contained Kaggle agent for **Maze Crawler** with:

- persistent wall and node memory,
- role-based unit behavior,
- northward pressure to stay ahead of the scroll,
- scouting toward the mirrored enemy side,
- miner-to-mine conversion when a node is found,
- cautious collision avoidance and energy consolidation.

The goal is a strong, readable baseline that is ready to submit from Kaggle Notebook Editor.

## Strategy

The policy leans into the most important elements of the ruleset:

1. **Factory survival first** — keep the factory moving north unless the board is safe enough to justify building.
2. **Early exploration** — build scouts early to reveal walls, nodes, and the enemy half.
3. **Map memory** — remember walls permanently, plus discovered nodes and mines.
4. **Economic pivots** — convert miners into mines on discovered nodes, then let friendly units refill on those mines.
5. **Low-risk movement** — avoid friendly collisions and only attack adjacent enemies that are clearly weaker.
6. **Path correction** — workers remove north walls when the route is blocked.

The result is not a random walk. Each unit has a different purpose and the factory adapts to pressure from the scroll.

In [1]:

from collections import deque
from math import inf
import random

# Shared state across turns inside Kaggle's notebook runtime.
STATE = {
    "turn": 0,
    "walls": {},          # (col, row) -> wall bitfield
    "nodes": set(),       # discovered mining nodes
    "mines": {},          # (col, row) -> [energy, maxEnergy, owner]
    "enemy_factory": None,
    "my_factory": None,
    "enemy_seen": {},     # uid -> last visible enemy data
}

TYPE_FACTORY = 0
TYPE_SCOUT = 1
TYPE_WORKER = 2
TYPE_MINER = 3

BIT_N, BIT_E, BIT_S, BIT_W = 1, 2, 4, 8

DIRS = {
    "NORTH": (0, 1, BIT_N),
    "EAST": (1, 0, BIT_E),
    "SOUTH": (0, -1, BIT_S),
    "WEST": (-1, 0, BIT_W),
}
DIR_ORDER = ("NORTH", "EAST", "WEST", "SOUTH")
OPPOSITE_BIT = {
    "NORTH": BIT_S,
    "EAST": BIT_W,
    "SOUTH": BIT_N,
    "WEST": BIT_E,
}
MOVE_ACTIONS = set(DIRS)

def parse_key(key):
    c, r = key.split(",")
    return int(c), int(r)

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def in_bounds(c, r, obs, config):
    return 0 <= c < config.width and obs.southBound <= r <= obs.northBound

def visible_range_rows(obs):
    return range(obs.southBound, obs.northBound + 1)

def update_memory(obs, config):
    STATE["turn"] += 1

    # Remember walls permanently.
    width = config.width
    for r in visible_range_rows(obs):
        base = (r - obs.southBound) * width
        if base < 0 or base >= len(obs.walls):
            continue
        for c in range(width):
            idx = base + c
            if idx >= len(obs.walls):
                break
            w = obs.walls[idx]
            if w != -1:
                STATE["walls"][(c, r)] = int(w)

    # Remember mining nodes we have ever seen.
    for key in getattr(obs, "miningNodes", {}) or {}:
        STATE["nodes"].add(parse_key(key))

    # Mines are persistent in the official observation.
    for key, val in getattr(obs, "mines", {}).items():
        STATE["mines"][parse_key(key)] = list(val)

    # Track enemy sightings for opportunistic targeting.
    STATE["enemy_seen"].clear()
    for uid, data in obs.robots.items():
        if data[4] != obs.player:
            STATE["enemy_seen"][uid] = tuple(data)

    # Track our factory if visible.
    my_factories = []
    for uid, data in obs.robots.items():
        if data[4] == obs.player and data[0] == TYPE_FACTORY:
            my_factories.append((uid, data))
    if my_factories:
        uid, data = my_factories[0]
        STATE["my_factory"] = (uid, data[1], data[2], data[3])

    # Track an enemy factory if visible, otherwise keep the mirrored guess.
    visible_enemy_factory = None
    for uid, data in obs.robots.items():
        if data[4] != obs.player and data[0] == TYPE_FACTORY:
            visible_enemy_factory = (uid, data[1], data[2], data[3])
            break
    if visible_enemy_factory:
        STATE["enemy_factory"] = visible_enemy_factory

def get_my_units(obs):
    units = []
    for uid, data in obs.robots.items():
        if data[4] == obs.player:
            units.append((uid, data))
    # Stronger units first so reservations prefer them.
    units.sort(key=lambda item: (-item[1][0], -item[1][3], item[0]))
    return units

def wall_bits_at(c, r):
    return STATE["walls"].get((c, r))

def blocked(c, r, direction, obs, config):
    dc, dr, bit = DIRS[direction]
    nc, nr = c + dc, r + dr
    if not in_bounds(nc, nr, obs, config):
        return True

    here = wall_bits_at(c, r)
    if here is not None and (here & bit):
        return True

    there = wall_bits_at(nc, nr)
    if there is not None and (there & OPPOSITE_BIT[direction]):
        return True

    return False

def bfs_first_step(start, goal, obs, config):
    if start == goal:
        return None

    q = deque([start])
    prev = {start: None}
    prev_dir = {start: None}
    visited = {start}

    best = start
    best_score = manhattan(start, goal)

    while q:
        cur = q.popleft()
        score = manhattan(cur, goal)
        if score < best_score:
            best = cur
            best_score = score

        if cur == goal:
            best = cur
            break

        for d in DIR_ORDER:
            dc, dr, _ = DIRS[d]
            nxt = (cur[0] + dc, cur[1] + dr)
            if nxt in visited:
                continue
            if not in_bounds(nxt[0], nxt[1], obs, config):
                continue
            if blocked(cur[0], cur[1], d, obs, config):
                continue
            visited.add(nxt)
            prev[nxt] = cur
            prev_dir[nxt] = d
            q.append(nxt)

    target = goal if goal in visited else best
    if target == start:
        return None

    cur = target
    while prev[cur] != start:
        cur = prev[cur]
        if cur is None:
            return None
    return prev_dir[cur] if prev[cur] == start else None

def target_to_step(start, target, obs, config):
    step = bfs_first_step(start, target, obs, config)
    if step is not None:
        return step

    # Greedy fallback: choose a legal move that gets closer to the target.
    candidates = []
    for d in DIR_ORDER:
        dc, dr, _ = DIRS[d]
        nxt = (start[0] + dc, start[1] + dr)
        if not in_bounds(nxt[0], nxt[1], obs, config):
            continue
        if blocked(start[0], start[1], d, obs, config):
            continue
        candidates.append((manhattan(nxt, target), d))
    if not candidates:
        return "IDLE"
    candidates.sort()
    return candidates[0][1]

def visible_enemy_targets(obs):
    out = []
    for uid, data in obs.robots.items():
        if data[4] != obs.player:
            out.append((uid, data))
    return out

def team_energy(obs):
    return sum(data[3] for data in obs.robots.values() if data[4] == obs.player)

def count_units(obs, robot_type=None):
    if robot_type is None:
        return sum(1 for data in obs.robots.values() if data[4] == obs.player)
    return sum(1 for data in obs.robots.values() if data[4] == obs.player and data[0] == robot_type)

def mirrored_enemy_guess(obs, config):
    if STATE["enemy_factory"] is not None:
        _, c, r, _ = STATE["enemy_factory"]
        return (c, r)
    if STATE["my_factory"] is None:
        return None
    _, c, r, _ = STATE["my_factory"]
    return (config.width - 1 - c, r)

def nearby_visible_crystals(obs):
    crystals = []
    for key, energy in (getattr(obs, "crystals", {}) or {}).items():
        crystals.append((parse_key(key), energy))
    return crystals

def visible_mining_nodes(obs):
    return [parse_key(k) for k in (getattr(obs, "miningNodes", {}) or {})]

def maybe_transfer(uid, data, obs, config, actions, reserved):
    """Consolidate energy into the nearest useful friendly if adjacency makes it free."""
    rtype, c, r, energy = data[0], data[1], data[2], data[3]
    if energy <= 1:
        return False

    best_target = None
    best_score = -inf
    for other_uid, other in obs.robots.items():
        if other_uid == uid or other[4] != obs.player:
            continue
        oc, orow, oenergy = other[1], other[2], other[3]
        if abs(oc - c) + abs(orow - r) != 1:
            continue
        # Prefer factories and low-energy units, and don't split a healthy scout/worker too early.
        score = 0.0
        if other[0] == TYPE_FACTORY:
            score += 1000.0
        elif other[0] == TYPE_MINER:
            score += 250.0
        elif other[0] == TYPE_WORKER:
            score += 150.0
        else:
            score += 50.0
        score += max(0, 200 - oenergy)
        score += min(50, energy)
        if score > best_score:
            best_score = score
            best_target = (other_uid, other)

    if best_target is None:
        return False

    _, other = best_target
    oc, orow, oenergy = other[1], other[2], other[3]
    if other[0] == TYPE_FACTORY and energy >= 75 and oenergy < 900:
        if oc == c + 1:
            actions[uid] = "TRANSFER_EAST"
            reserved.add((c, r))
            return True
        if oc == c - 1:
            actions[uid] = "TRANSFER_WEST"
            reserved.add((c, r))
            return True
        if orow == r + 1:
            actions[uid] = "TRANSFER_NORTH"
            reserved.add((c, r))
            return True
        if orow == r - 1:
            actions[uid] = "TRANSFER_SOUTH"
            reserved.add((c, r))
            return True

    if rtype in (TYPE_WORKER, TYPE_MINER) and energy >= 150 and oenergy < 120:
        if oc == c + 1:
            actions[uid] = "TRANSFER_EAST"
            reserved.add((c, r))
            return True
        if oc == c - 1:
            actions[uid] = "TRANSFER_WEST"
            reserved.add((c, r))
            return True
        if orow == r + 1:
            actions[uid] = "TRANSFER_NORTH"
            reserved.add((c, r))
            return True
        if orow == r - 1:
            actions[uid] = "TRANSFER_SOUTH"
            reserved.add((c, r))
            return True

    return False


In [2]:

strength_rank = {
    TYPE_FACTORY: 4,
    TYPE_MINER: 3,
    TYPE_WORKER: 2,
    TYPE_SCOUT: 1,
}

max_energy = {
    TYPE_SCOUT: 100,
    TYPE_WORKER: 300,
    TYPE_MINER: 500,
}

def current_occupants(obs):
    occ = {}
    for uid, data in obs.robots.items():
        cell = (data[1], data[2])
        occ.setdefault(cell, []).append((uid, data))
    return occ

def best_attack_step(uid, data, obs, config, occupied):
    """Return a move direction into a weaker adjacent enemy, or None."""
    if data[5] != 0:  # move cooldown
        return None

    c, r = data[1], data[2]
    my_strength = strength_rank[data[0]]

    best = None
    best_score = -inf

    for d in DIR_ORDER:
        dc, dr, _ = DIRS[d]
        nxt = (c + dc, r + dr)
        if not in_bounds(nxt[0], nxt[1], obs, config):
            continue
        if blocked(c, r, d, obs, config):
            continue

        occupants = occupied.get(nxt, [])
        allies = [o for o in occupants if o[1][4] == obs.player]
        enemies = [o for o in occupants if o[1][4] != obs.player]
        if allies or not enemies:
            continue

        enemy_strength = max(strength_rank[o[1][0]] for o in enemies)
        if my_strength <= enemy_strength:
            continue

        # Prefer attacks on isolated targets and higher-value enemy pieces.
        score = 10.0 * (my_strength - enemy_strength)
        score += 0.5 * len(enemies)
        if any(o[1][0] == TYPE_FACTORY for o in enemies):
            score += 100.0
        if any(o[1][0] == TYPE_MINER for o in enemies):
            score += 20.0
        if score > best_score:
            best_score = score
            best = d

    return best

def on_friendly_mine(uid, data, obs):
    cell = (data[1], data[2])
    mine = STATE["mines"].get(cell)
    return bool(mine and mine[2] == obs.player)

def choose_scout_target(uid, data, obs, config):
    c, r = data[1], data[2]

    # Prioritize nearby crystals if they are clearly worth detouring for.
    crystals = nearby_visible_crystals(obs)
    if crystals:
        best = None
        best_score = -inf
        for cell, energy in crystals:
            dist = manhattan((c, r), cell)
            score = energy / max(1, dist)
            # Gentle north bias so scouts still keep climbing.
            score += 0.15 * (cell[1] - r)
            if score > best_score:
                best_score = score
                best = cell
        if best is not None and best_score >= 6:
            return best

    guess = mirrored_enemy_guess(obs, config)
    if guess is not None:
        return guess

    # Default: keep pushing north in the current column to uncover the maze.
    return (c, min(obs.northBound, r + 6))

def choose_worker_target(uid, data, obs, config):
    c, r = data[1], data[2]

    # Workers are utility pieces: keep heading north, but also favor discovered nodes.
    nodes = [p for p in STATE["nodes"] if p not in STATE["mines"]]
    if nodes:
        # Prefer accessible nodes that are closest in Manhattan distance.
        target, dist = nearest_point((c, r), nodes)
        if target is not None and dist <= 12:
            return target

    guess = mirrored_enemy_guess(obs, config)
    if guess is not None:
        return guess
    return (c, min(obs.northBound, r + 6))

def choose_miner_target(uid, data, obs, config):
    c, r = data[1], data[2]
    nodes = [p for p in STATE["nodes"] if p not in STATE["mines"]]
    if nodes:
        target, _ = nearest_point((c, r), nodes)
        if target is not None:
            return target
    guess = mirrored_enemy_guess(obs, config)
    if guess is not None:
        return guess
    return (c, min(obs.northBound, r + 6))

def remove_direction_if_blocked(uid, data, obs, config, actions, reserved):
    c, r = data[1], data[2]
    build_cd = data[7] if len(data) > 7 else 0
    if build_cd != 0 or data[3] < getattr(config, "wallRemoveCost", 100):
        return False

    # Open the path north if we know there is a north wall from either side.
    w_here = wall_bits_at(c, r)
    w_above = wall_bits_at(c, r + 1) if in_bounds(c, r + 1, obs, config) else None
    if (w_here is not None and (w_here & BIT_N)) or (w_above is not None and (w_above & BIT_S)):
        actions[uid] = "REMOVE_NORTH"
        reserved.add((c, r))
        return True

    return False

def decide_factory(uid, data, obs, config, actions, reserved, occupied, rng):
    c, r, energy = data[1], data[2], data[3]
    move_cd = data[5] if len(data) > 5 else 0
    jump_cd = data[6] if len(data) > 6 else 0
    build_cd = data[7] if len(data) > 7 else 0

    safety_gap = r - obs.southBound
    scout_count = count_units(obs, TYPE_SCOUT)
    worker_count = count_units(obs, TYPE_WORKER)
    miner_count = count_units(obs, TYPE_MINER)
    enemy_visible = len(visible_enemy_targets(obs)) > 0
    spawn = (c, r + 1)
    spawn_clear = in_bounds(spawn[0], spawn[1], obs, config) and spawn not in occupied

    # Protect against the scroll first.
    if safety_gap <= 3:
        if jump_cd == 0 and r + 2 <= obs.northBound:
            actions[uid] = "JUMP_NORTH"
            reserved.add((c, r + 2))
            return True
        if move_cd == 0:
            step = target_to_step((c, r), (c, min(obs.northBound, r + 4)), obs, config)
            if step in MOVE_ACTIONS:
                dc, dr, _ = DIRS[step]
                reserved.add((c + dc, r + dr))
                actions[uid] = step
                return True
        actions[uid] = "IDLE"
        reserved.add((c, r))
        return True

    # Build only when the spawn cell is open and the factory is not under immediate pressure.
    if build_cd == 0 and spawn_clear:
        if scout_count < 1 and energy >= getattr(config, "scoutCost", 50):
            actions[uid] = "BUILD_SCOUT"
            reserved.add(spawn)
            return True

        if STATE["nodes"] and miner_count < 1 and energy >= getattr(config, "minerCost", 300) + 120:
            actions[uid] = "BUILD_MINER"
            reserved.add(spawn)
            return True

        if enemy_visible and worker_count < 2 and energy >= getattr(config, "workerCost", 200) + 120:
            actions[uid] = "BUILD_WORKER"
            reserved.add(spawn)
            return True

        if scout_count < 2 and energy >= getattr(config, "scoutCost", 50) + 75:
            actions[uid] = "BUILD_SCOUT"
            reserved.add(spawn)
            return True

        if worker_count < 1 and energy >= getattr(config, "workerCost", 200) + 120:
            actions[uid] = "BUILD_WORKER"
            reserved.add(spawn)
            return True

        if STATE["nodes"] and miner_count < 2 and energy >= getattr(config, "minerCost", 300) + 220:
            actions[uid] = "BUILD_MINER"
            reserved.add(spawn)
            return True

    # Movement / jump fallback.
    if move_cd == 0:
        step = target_to_step((c, r), (c, min(obs.northBound, r + 5)), obs, config)
        if step in MOVE_ACTIONS:
            dc, dr, _ = DIRS[step]
            nxt = (c + dc, r + dr)
            reserved.add(nxt)
            actions[uid] = step
            return True

    if jump_cd == 0 and r + 2 <= obs.northBound and safety_gap <= 8:
        actions[uid] = "JUMP_NORTH"
        reserved.add((c, r + 2))
        return True

    actions[uid] = "IDLE"
    reserved.add((c, r))
    return True

def decide_nonfactory(uid, data, obs, config, actions, reserved, occupied, rng):
    rtype, c, r, energy = data[0], data[1], data[2], data[3]
    move_cd = data[5] if len(data) > 5 else 0
    build_cd = data[7] if len(data) > 7 else 0

    # Miner conversion is the most important special action.
    if rtype == TYPE_MINER and build_cd == 0 and energy >= getattr(config, "transformCost", 100) + 1:
        if (c, r) in STATE["nodes"]:
            actions[uid] = "TRANSFORM"
            reserved.add((c, r))
            return True

    # Transfers are useful for consolidating energy, especially near factories.
    if maybe_transfer(uid, data, obs, config, actions, reserved):
        return True

    # Worker wall removal to keep northbound access open.
    if rtype == TYPE_WORKER and remove_direction_if_blocked(uid, data, obs, config, actions, reserved):
        return True

    # Enemy-adjacent crush moves.
    attack_step = best_attack_step(uid, data, obs, config, occupied)
    if attack_step is not None:
        dc, dr, _ = DIRS[attack_step]
        actions[uid] = attack_step
        reserved.add((c + dc, r + dr))
        return True

    # If we are standing on a friendly mine, let the mine refill us.
    if on_friendly_mine(uid, data, obs) and rtype != TYPE_FACTORY:
        cap = max_energy.get(rtype, 10 ** 9)
        if energy < cap - 5:
            actions[uid] = "IDLE"
            reserved.add((c, r))
            return True

    # If the action is cooling down or we are already healthy on a mine, stay put.
    if move_cd != 0:
        actions[uid] = "IDLE"
        reserved.add((c, r))
        return True

    # Role-specific travel targets.
    if rtype == TYPE_SCOUT:
        target = choose_scout_target(uid, data, obs, config)
    elif rtype == TYPE_WORKER:
        target = choose_worker_target(uid, data, obs, config)
    elif rtype == TYPE_MINER:
        target = choose_miner_target(uid, data, obs, config)
    else:
        target = (c, min(obs.northBound, r + 4))

    step = target_to_step((c, r), target, obs, config)
    if step in MOVE_ACTIONS:
        dc, dr, _ = DIRS[step]
        nxt = (c + dc, r + dr)
        # Avoid moving into our own units or into already-reserved cells.
        if nxt not in reserved:
            occup = occupied.get(nxt, [])
            if not any(o[1][4] == obs.player for o in occup):
                actions[uid] = step
                reserved.add(nxt)
                return True

    # If no move is good, idling is often better than a bad collision.
    actions[uid] = "IDLE"
    reserved.add((c, r))
    return True



def agent(obs, config):
    update_memory(obs, config)

    actions = {}
    reserved = set()
    occupied = current_occupants(obs)
    rng = random.Random(7919 + STATE["turn"] * 17 + obs.player)

    # Factories decide first so their spawn cell can be reserved before movers plan.
    for uid, data in obs.robots.items():
        if data[4] == obs.player and data[0] == TYPE_FACTORY:
            decide_factory(uid, data, obs, config, actions, reserved, occupied, rng)

    # Then handle mobile pieces, preferring stronger units and preserving collisions.
    others = [
        (uid, data)
        for uid, data in obs.robots.items()
        if data[4] == obs.player and data[0] != TYPE_FACTORY
    ]
    others.sort(key=lambda item: (-strength_rank[item[1][0]], -item[1][3], item[0]))

    for uid, data in others:
        if uid in actions:
            continue
        decide_nonfactory(uid, data, obs, config, actions, reserved, occupied, rng)

    return actions


## Smoke test

If `kaggle_environments` is available in the runtime, the cell below runs a quick local match against `random` and prints the final rewards.

In [3]:
try:
    from kaggle_environments import make

    env = make("crawl", configuration={"randomSeed": 42}, debug=True)
    env.run([agent, "random"])

    final = env.steps[-1]
    print([(i, s.reward, s.status) for i, s in enumerate(final)])
    # env.render(mode="ipython", width=800, height=800)
except Exception as e:
    print("Smoke test skipped:", type(e).__name__, e)

[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO: Successfully loaded OpenSpiel environments: 16.
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_backgammon
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_checkers
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_chess
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_connect_four
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_gin_rummy
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_go
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_goofspiel
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hearts
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_hex
[kaggle_environments.envs.open_spiel_env.open_spiel_env] INFO:    open_spiel_matching_pennies_3p
[kaggle_environments.envs.

## Submission output

In [4]:
# Write a standalone submission file for Kaggle Notebook submissions.
submission_source = r'''from collections import deque
from math import inf
import random

# Shared state across turns inside Kaggle's notebook runtime.
STATE = {
    "turn": 0,
    "walls": {},          # (col, row) -> wall bitfield
    "nodes": set(),       # discovered mining nodes
    "mines": {},          # (col, row) -> [energy, maxEnergy, owner]
    "enemy_factory": None,
    "my_factory": None,
    "enemy_seen": {},     # uid -> last visible enemy data
}

TYPE_FACTORY = 0
TYPE_SCOUT = 1
TYPE_WORKER = 2
TYPE_MINER = 3

BIT_N, BIT_E, BIT_S, BIT_W = 1, 2, 4, 8

DIRS = {
    "NORTH": (0, 1, BIT_N),
    "EAST": (1, 0, BIT_E),
    "SOUTH": (0, -1, BIT_S),
    "WEST": (-1, 0, BIT_W),
}
DIR_ORDER = ("NORTH", "EAST", "WEST", "SOUTH")
OPPOSITE_BIT = {
    "NORTH": BIT_S,
    "EAST": BIT_W,
    "SOUTH": BIT_N,
    "WEST": BIT_E,
}
MOVE_ACTIONS = set(DIRS)

def parse_key(key):
    c, r = key.split(",")
    return int(c), int(r)

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def in_bounds(c, r, obs, config):
    return 0 <= c < config.width and obs.southBound <= r <= obs.northBound

def visible_range_rows(obs):
    return range(obs.southBound, obs.northBound + 1)

def update_memory(obs, config):
    STATE["turn"] += 1

    # Remember walls permanently.
    width = config.width
    for r in visible_range_rows(obs):
        base = (r - obs.southBound) * width
        if base < 0 or base >= len(obs.walls):
            continue
        for c in range(width):
            idx = base + c
            if idx >= len(obs.walls):
                break
            w = obs.walls[idx]
            if w != -1:
                STATE["walls"][(c, r)] = int(w)

    # Remember mining nodes we have ever seen.
    for key in getattr(obs, "miningNodes", {}) or {}:
        STATE["nodes"].add(parse_key(key))

    # Mines are persistent in the official observation.
    for key, val in getattr(obs, "mines", {}).items():
        STATE["mines"][parse_key(key)] = list(val)

    # Track enemy sightings for opportunistic targeting.
    STATE["enemy_seen"].clear()
    for uid, data in obs.robots.items():
        if data[4] != obs.player:
            STATE["enemy_seen"][uid] = tuple(data)

    # Track our factory if visible.
    my_factories = []
    for uid, data in obs.robots.items():
        if data[4] == obs.player and data[0] == TYPE_FACTORY:
            my_factories.append((uid, data))
    if my_factories:
        uid, data = my_factories[0]
        STATE["my_factory"] = (uid, data[1], data[2], data[3])

    # Track an enemy factory if visible, otherwise keep the mirrored guess.
    visible_enemy_factory = None
    for uid, data in obs.robots.items():
        if data[4] != obs.player and data[0] == TYPE_FACTORY:
            visible_enemy_factory = (uid, data[1], data[2], data[3])
            break
    if visible_enemy_factory:
        STATE["enemy_factory"] = visible_enemy_factory

def get_my_units(obs):
    units = []
    for uid, data in obs.robots.items():
        if data[4] == obs.player:
            units.append((uid, data))
    # Stronger units first so reservations prefer them.
    units.sort(key=lambda item: (-item[1][0], -item[1][3], item[0]))
    return units

def wall_bits_at(c, r):
    return STATE["walls"].get((c, r))

def blocked(c, r, direction, obs, config):
    dc, dr, bit = DIRS[direction]
    nc, nr = c + dc, r + dr
    if not in_bounds(nc, nr, obs, config):
        return True

    here = wall_bits_at(c, r)
    if here is not None and (here & bit):
        return True

    there = wall_bits_at(nc, nr)
    if there is not None and (there & OPPOSITE_BIT[direction]):
        return True

    return False

def bfs_first_step(start, goal, obs, config):
    if start == goal:
        return None

    q = deque([start])
    prev = {start: None}
    prev_dir = {start: None}
    visited = {start}

    best = start
    best_score = manhattan(start, goal)

    while q:
        cur = q.popleft()
        score = manhattan(cur, goal)
        if score < best_score:
            best = cur
            best_score = score

        if cur == goal:
            best = cur
            break

        for d in DIR_ORDER:
            dc, dr, _ = DIRS[d]
            nxt = (cur[0] + dc, cur[1] + dr)
            if nxt in visited:
                continue
            if not in_bounds(nxt[0], nxt[1], obs, config):
                continue
            if blocked(cur[0], cur[1], d, obs, config):
                continue
            visited.add(nxt)
            prev[nxt] = cur
            prev_dir[nxt] = d
            q.append(nxt)

    target = goal if goal in visited else best
    if target == start:
        return None

    cur = target
    while prev[cur] != start:
        cur = prev[cur]
        if cur is None:
            return None
    return prev_dir[cur] if prev[cur] == start else None

def target_to_step(start, target, obs, config):
    step = bfs_first_step(start, target, obs, config)
    if step is not None:
        return step

    # Greedy fallback: choose a legal move that gets closer to the target.
    candidates = []
    for d in DIR_ORDER:
        dc, dr, _ = DIRS[d]
        nxt = (start[0] + dc, start[1] + dr)
        if not in_bounds(nxt[0], nxt[1], obs, config):
            continue
        if blocked(start[0], start[1], d, obs, config):
            continue
        candidates.append((manhattan(nxt, target), d))
    if not candidates:
        return "IDLE"
    candidates.sort()
    return candidates[0][1]

def visible_enemy_targets(obs):
    out = []
    for uid, data in obs.robots.items():
        if data[4] != obs.player:
            out.append((uid, data))
    return out

def team_energy(obs):
    return sum(data[3] for data in obs.robots.values() if data[4] == obs.player)

def count_units(obs, robot_type=None):
    if robot_type is None:
        return sum(1 for data in obs.robots.values() if data[4] == obs.player)
    return sum(1 for data in obs.robots.values() if data[4] == obs.player and data[0] == robot_type)

def mirrored_enemy_guess(obs, config):
    if STATE["enemy_factory"] is not None:
        _, c, r, _ = STATE["enemy_factory"]
        return (c, r)
    if STATE["my_factory"] is None:
        return None
    _, c, r, _ = STATE["my_factory"]
    return (config.width - 1 - c, r)

def nearby_visible_crystals(obs):
    crystals = []
    for key, energy in (getattr(obs, "crystals", {}) or {}).items():
        crystals.append((parse_key(key), energy))
    return crystals

def visible_mining_nodes(obs):
    return [parse_key(k) for k in (getattr(obs, "miningNodes", {}) or {})]

def maybe_transfer(uid, data, obs, config, actions, reserved):
    """Consolidate energy into the nearest useful friendly if adjacency makes it free."""
    rtype, c, r, energy = data[0], data[1], data[2], data[3]
    if energy <= 1:
        return False

    best_target = None
    best_score = -inf
    for other_uid, other in obs.robots.items():
        if other_uid == uid or other[4] != obs.player:
            continue
        oc, orow, oenergy = other[1], other[2], other[3]
        if abs(oc - c) + abs(orow - r) != 1:
            continue
        # Prefer factories and low-energy units, and don't split a healthy scout/worker too early.
        score = 0.0
        if other[0] == TYPE_FACTORY:
            score += 1000.0
        elif other[0] == TYPE_MINER:
            score += 250.0
        elif other[0] == TYPE_WORKER:
            score += 150.0
        else:
            score += 50.0
        score += max(0, 200 - oenergy)
        score += min(50, energy)
        if score > best_score:
            best_score = score
            best_target = (other_uid, other)

    if best_target is None:
        return False

    _, other = best_target
    oc, orow, oenergy = other[1], other[2], other[3]
    if other[0] == TYPE_FACTORY and energy >= 75 and oenergy < 900:
        if oc == c + 1:
            actions[uid] = "TRANSFER_EAST"
            reserved.add((c, r))
            return True
        if oc == c - 1:
            actions[uid] = "TRANSFER_WEST"
            reserved.add((c, r))
            return True
        if orow == r + 1:
            actions[uid] = "TRANSFER_NORTH"
            reserved.add((c, r))
            return True
        if orow == r - 1:
            actions[uid] = "TRANSFER_SOUTH"
            reserved.add((c, r))
            return True

    if rtype in (TYPE_WORKER, TYPE_MINER) and energy >= 150 and oenergy < 120:
        if oc == c + 1:
            actions[uid] = "TRANSFER_EAST"
            reserved.add((c, r))
            return True
        if oc == c - 1:
            actions[uid] = "TRANSFER_WEST"
            reserved.add((c, r))
            return True
        if orow == r + 1:
            actions[uid] = "TRANSFER_NORTH"
            reserved.add((c, r))
            return True
        if orow == r - 1:
            actions[uid] = "TRANSFER_SOUTH"
            reserved.add((c, r))
            return True

    return False



strength_rank = {
    TYPE_FACTORY: 4,
    TYPE_MINER: 3,
    TYPE_WORKER: 2,
    TYPE_SCOUT: 1,
}

max_energy = {
    TYPE_SCOUT: 100,
    TYPE_WORKER: 300,
    TYPE_MINER: 500,
}

def current_occupants(obs):
    occ = {}
    for uid, data in obs.robots.items():
        cell = (data[1], data[2])
        occ.setdefault(cell, []).append((uid, data))
    return occ

def best_attack_step(uid, data, obs, config, occupied):
    """Return a move direction into a weaker adjacent enemy, or None."""
    if data[5] != 0:  # move cooldown
        return None

    c, r = data[1], data[2]
    my_strength = strength_rank[data[0]]

    best = None
    best_score = -inf

    for d in DIR_ORDER:
        dc, dr, _ = DIRS[d]
        nxt = (c + dc, r + dr)
        if not in_bounds(nxt[0], nxt[1], obs, config):
            continue
        if blocked(c, r, d, obs, config):
            continue

        occupants = occupied.get(nxt, [])
        allies = [o for o in occupants if o[1][4] == obs.player]
        enemies = [o for o in occupants if o[1][4] != obs.player]
        if allies or not enemies:
            continue

        enemy_strength = max(strength_rank[o[1][0]] for o in enemies)
        if my_strength <= enemy_strength:
            continue

        # Prefer attacks on isolated targets and higher-value enemy pieces.
        score = 10.0 * (my_strength - enemy_strength)
        score += 0.5 * len(enemies)
        if any(o[1][0] == TYPE_FACTORY for o in enemies):
            score += 100.0
        if any(o[1][0] == TYPE_MINER for o in enemies):
            score += 20.0
        if score > best_score:
            best_score = score
            best = d

    return best

def on_friendly_mine(uid, data, obs):
    cell = (data[1], data[2])
    mine = STATE["mines"].get(cell)
    return bool(mine and mine[2] == obs.player)

def choose_scout_target(uid, data, obs, config):
    c, r = data[1], data[2]

    # Prioritize nearby crystals if they are clearly worth detouring for.
    crystals = nearby_visible_crystals(obs)
    if crystals:
        best = None
        best_score = -inf
        for cell, energy in crystals:
            dist = manhattan((c, r), cell)
            score = energy / max(1, dist)
            # Gentle north bias so scouts still keep climbing.
            score += 0.15 * (cell[1] - r)
            if score > best_score:
                best_score = score
                best = cell
        if best is not None and best_score >= 6:
            return best

    guess = mirrored_enemy_guess(obs, config)
    if guess is not None:
        return guess

    # Default: keep pushing north in the current column to uncover the maze.
    return (c, min(obs.northBound, r + 6))

def choose_worker_target(uid, data, obs, config):
    c, r = data[1], data[2]

    # Workers are utility pieces: keep heading north, but also favor discovered nodes.
    nodes = [p for p in STATE["nodes"] if p not in STATE["mines"]]
    if nodes:
        # Prefer accessible nodes that are closest in Manhattan distance.
        target, dist = nearest_point((c, r), nodes)
        if target is not None and dist <= 12:
            return target

    guess = mirrored_enemy_guess(obs, config)
    if guess is not None:
        return guess
    return (c, min(obs.northBound, r + 6))

def choose_miner_target(uid, data, obs, config):
    c, r = data[1], data[2]
    nodes = [p for p in STATE["nodes"] if p not in STATE["mines"]]
    if nodes:
        target, _ = nearest_point((c, r), nodes)
        if target is not None:
            return target
    guess = mirrored_enemy_guess(obs, config)
    if guess is not None:
        return guess
    return (c, min(obs.northBound, r + 6))

def remove_direction_if_blocked(uid, data, obs, config, actions, reserved):
    c, r = data[1], data[2]
    build_cd = data[7] if len(data) > 7 else 0
    if build_cd != 0 or data[3] < getattr(config, "wallRemoveCost", 100):
        return False

    # Open the path north if we know there is a north wall from either side.
    w_here = wall_bits_at(c, r)
    w_above = wall_bits_at(c, r + 1) if in_bounds(c, r + 1, obs, config) else None
    if (w_here is not None and (w_here & BIT_N)) or (w_above is not None and (w_above & BIT_S)):
        actions[uid] = "REMOVE_NORTH"
        reserved.add((c, r))
        return True

    return False

def decide_factory(uid, data, obs, config, actions, reserved, occupied, rng):
    c, r, energy = data[1], data[2], data[3]
    move_cd = data[5] if len(data) > 5 else 0
    jump_cd = data[6] if len(data) > 6 else 0
    build_cd = data[7] if len(data) > 7 else 0

    safety_gap = r - obs.southBound
    scout_count = count_units(obs, TYPE_SCOUT)
    worker_count = count_units(obs, TYPE_WORKER)
    miner_count = count_units(obs, TYPE_MINER)
    enemy_visible = len(visible_enemy_targets(obs)) > 0
    spawn = (c, r + 1)
    spawn_clear = in_bounds(spawn[0], spawn[1], obs, config) and spawn not in occupied

    # Protect against the scroll first.
    if safety_gap <= 3:
        if jump_cd == 0 and r + 2 <= obs.northBound:
            actions[uid] = "JUMP_NORTH"
            reserved.add((c, r + 2))
            return True
        if move_cd == 0:
            step = target_to_step((c, r), (c, min(obs.northBound, r + 4)), obs, config)
            if step in MOVE_ACTIONS:
                dc, dr, _ = DIRS[step]
                reserved.add((c + dc, r + dr))
                actions[uid] = step
                return True
        actions[uid] = "IDLE"
        reserved.add((c, r))
        return True

    # Build only when the spawn cell is open and the factory is not under immediate pressure.
    if build_cd == 0 and spawn_clear:
        if scout_count < 1 and energy >= getattr(config, "scoutCost", 50):
            actions[uid] = "BUILD_SCOUT"
            reserved.add(spawn)
            return True

        if STATE["nodes"] and miner_count < 1 and energy >= getattr(config, "minerCost", 300) + 120:
            actions[uid] = "BUILD_MINER"
            reserved.add(spawn)
            return True

        if enemy_visible and worker_count < 2 and energy >= getattr(config, "workerCost", 200) + 120:
            actions[uid] = "BUILD_WORKER"
            reserved.add(spawn)
            return True

        if scout_count < 2 and energy >= getattr(config, "scoutCost", 50) + 75:
            actions[uid] = "BUILD_SCOUT"
            reserved.add(spawn)
            return True

        if worker_count < 1 and energy >= getattr(config, "workerCost", 200) + 120:
            actions[uid] = "BUILD_WORKER"
            reserved.add(spawn)
            return True

        if STATE["nodes"] and miner_count < 2 and energy >= getattr(config, "minerCost", 300) + 220:
            actions[uid] = "BUILD_MINER"
            reserved.add(spawn)
            return True

    # Movement / jump fallback.
    if move_cd == 0:
        step = target_to_step((c, r), (c, min(obs.northBound, r + 5)), obs, config)
        if step in MOVE_ACTIONS:
            dc, dr, _ = DIRS[step]
            nxt = (c + dc, r + dr)
            reserved.add(nxt)
            actions[uid] = step
            return True

    if jump_cd == 0 and r + 2 <= obs.northBound and safety_gap <= 8:
        actions[uid] = "JUMP_NORTH"
        reserved.add((c, r + 2))
        return True

    actions[uid] = "IDLE"
    reserved.add((c, r))
    return True

def decide_nonfactory(uid, data, obs, config, actions, reserved, occupied, rng):
    rtype, c, r, energy = data[0], data[1], data[2], data[3]
    move_cd = data[5] if len(data) > 5 else 0
    build_cd = data[7] if len(data) > 7 else 0

    # Miner conversion is the most important special action.
    if rtype == TYPE_MINER and build_cd == 0 and energy >= getattr(config, "transformCost", 100) + 1:
        if (c, r) in STATE["nodes"]:
            actions[uid] = "TRANSFORM"
            reserved.add((c, r))
            return True

    # Transfers are useful for consolidating energy, especially near factories.
    if maybe_transfer(uid, data, obs, config, actions, reserved):
        return True

    # Worker wall removal to keep northbound access open.
    if rtype == TYPE_WORKER and remove_direction_if_blocked(uid, data, obs, config, actions, reserved):
        return True

    # Enemy-adjacent crush moves.
    attack_step = best_attack_step(uid, data, obs, config, occupied)
    if attack_step is not None:
        dc, dr, _ = DIRS[attack_step]
        actions[uid] = attack_step
        reserved.add((c + dc, r + dr))
        return True

    # If we are standing on a friendly mine, let the mine refill us.
    if on_friendly_mine(uid, data, obs) and rtype != TYPE_FACTORY:
        cap = max_energy.get(rtype, 10 ** 9)
        if energy < cap - 5:
            actions[uid] = "IDLE"
            reserved.add((c, r))
            return True

    # If the action is cooling down or we are already healthy on a mine, stay put.
    if move_cd != 0:
        actions[uid] = "IDLE"
        reserved.add((c, r))
        return True

    # Role-specific travel targets.
    if rtype == TYPE_SCOUT:
        target = choose_scout_target(uid, data, obs, config)
    elif rtype == TYPE_WORKER:
        target = choose_worker_target(uid, data, obs, config)
    elif rtype == TYPE_MINER:
        target = choose_miner_target(uid, data, obs, config)
    else:
        target = (c, min(obs.northBound, r + 4))

    step = target_to_step((c, r), target, obs, config)
    if step in MOVE_ACTIONS:
        dc, dr, _ = DIRS[step]
        nxt = (c + dc, r + dr)
        # Avoid moving into our own units or into already-reserved cells.
        if nxt not in reserved:
            occup = occupied.get(nxt, [])
            if not any(o[1][4] == obs.player for o in occup):
                actions[uid] = step
                reserved.add(nxt)
                return True

    # If no move is good, idling is often better than a bad collision.
    actions[uid] = "IDLE"
    reserved.add((c, r))
    return True



def agent(obs, config):
    update_memory(obs, config)

    actions = {}
    reserved = set()
    occupied = current_occupants(obs)
    rng = random.Random(7919 + STATE["turn"] * 17 + obs.player)

    # Factories decide first so their spawn cell can be reserved before movers plan.
    for uid, data in obs.robots.items():
        if data[4] == obs.player and data[0] == TYPE_FACTORY:
            decide_factory(uid, data, obs, config, actions, reserved, occupied, rng)

    # Then handle mobile pieces, preferring stronger units and preserving collisions.
    others = [
        (uid, data)
        for uid, data in obs.robots.items()
        if data[4] == obs.player and data[0] != TYPE_FACTORY
    ]
    others.sort(key=lambda item: (-strength_rank[item[1][0]], -item[1][3], item[0]))

    for uid, data in others:
        if uid in actions:
            continue
        decide_nonfactory(uid, data, obs, config, actions, reserved, occupied, rng)

    return actions
'''
from pathlib import Path

working = Path("/kaggle/working")
for name in ["submission.py", "main.py"]:
    out = working / name
    out.write_text(submission_source, encoding="utf-8")
    print(f"Wrote {out} ({out.stat().st_size} bytes)")

Wrote /kaggle/working/submission.py (19902 bytes)
Wrote /kaggle/working/main.py (19902 bytes)
